<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/Insight-Sogang-Univ/insight-15th/blob/main/advanced/template/session05/assignment_languagemodel.ipynb" target="_parent">
      <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
  </td>
</table>


# 🔍 이론 문제

### 1. SLM(통계적 언어모델)의 한계에 대해 설명해주세요.

**[1번 문제 정답]**

문맥이 길어지면 이전의 정보를 제대로 유지하거나 파악하기 어렵다는 한계.

### 2. BERT의 두 가지 사전학습 방식에 대해 간단히 설명해주세요



1. MLM: 문장 내 일부 단어 토큰을 무작위로 마스킹한 후, 모델이 양방향 문맥을 모두 고려하여 해당 빈칸에 들어갈 단어를 예측하도록 학습

2. NSP: 모델에 두 문장을 입력하고, 두 번째 문장이 첫 번째 문장 바로 다음에 이어지는 실제 문장인지 아닌지를 맞추도록 학습

### 3. LangChain, LangGraph에 대한 설명으로 옳지 않은 것은? <br>

a) LangChain은 LLM 파이프라인을 쉽게 구성하도록 하는 프레임워크이다.    
b) LangGraph는 노드와 엣지로 구성되며 순환 구조를 지원한다.    
c) LangChain은 조건에 따라 이전 단계로 되돌아가는 루프 로직 구현에 적합하다.      
d) LangSmith는 LLM 애플리케이션을 어떻게 사용하는지 모니터링한다.     




c

### 4. LLM이 지닌 한계와 RAG의 필요성에 대해 간단히 서술하세요.

**[4번 문제 정답]**

LLM은 학습 시점 이후의 최신 정보를 알 수 없고 내부 기업 데이터 등 특정 지식에 접근할 수 없음.
RAG는 사용자의 질문과 관련된 최신 정보나 외부 문서를 데이터베이스에서 직접 검색해 온 뒤 이를 바탕으로 답변을 생성하도록 함.

### 5. sLM의 장점과 sLM을 활용하기에 적합한 환경에 대해 설명해주세요.



sLM은 파라미터 수가 적어 모델 크기가 작기 때문에, 컴퓨팅 연산 자원과 메모리 소모가 적고 추론 속도가 빠르다는 게 장점.
따라서 스마트폰이나 노트북 등 로컬 기기에서 직접 구동하는할때 쓰임

# ✍ **언어 모델 과제**

### 🧮 GPU 사용하기

In [1]:
import torch
import tensorflow as tf

# PyTorch GPU 상태
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ [PyTorch] Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# TensorFlow GPU 상태
print("\n✅ [TensorFlow] GPU Devices:")
print(tf.config.list_physical_devices('GPU'))

✅ [PyTorch] Using device: cuda
GPU name: Tesla T4

✅ [TensorFlow] GPU Devices:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# 🔍 1. BERT

**BERT (Bidirectional Encoder Representations from Transformers)**
- 문장 분류(스팸 메일 탐지), 질의응답 시스템(챗봇, 검색 엔진), 번역(다국어 지원), 텍스트 요약(뉴스 요약) 등 다양한 NLP 작업에 사용
- 문맥을 양방향으로 이해하여 텍스트의 의미를 정밀하게 파악하며, 사전 학습된 모델을 기반으로 빠르게 응용 가능

### 1-1. 모델과 tokenizer 초기화
- tokenizer를 통해 문장을 토큰으로 나누고, 이를 정수 인덱스로 변환하여 모델이 이해할 수 있도록 함

In [2]:
!pip install -q --upgrade transformers huggingface_hub

from transformers import BertTokenizer, BertForMaskedLM
import torch

# BERT 모델과 tokenizer 초기화
tokenizer = BertTokenizer.from_pretrained('klue/bert-base') # base : 모델크기, (uncased : 소문자 학습)
model = BertForMaskedLM.from_pretrained('klue/bert-base') # base : 모델크기, (uncased : 소문자 학습)
# tokenizer : raw text를 개별 토큰으로 분리(Wordpiece). 토크나이저 사전에 따라 고유한 정수 인덱스(고정값)를 매핑함.

# tokenizer를 통해 생성된 정수 인덱스 : 텍스트를 모델이 읽을 수 있도록 숫자화
# Embedding vector : 단어의 의미적, 문맥적 특성을 모델링


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 1-2. 입력 텍스트 준비 및 토큰화
- [MASK] 토큰을 삽입한 문장을 설정하고, `tokenizer.tokenize()`로 텍스트 토큰화
- [MASK]를 활용하여 문맥 속에서 특정 단어를 추론하는 상황을 만들고, 이를 모델이 학습한 패턴과 비교하도록 함


In [3]:
# 테스트할 문장
text = "[CLS] 열여덟, 우리는 서로의 이름을 처음 불렀다. 그리고 스물 하나, 우린 [MASK]을 했다.[SEP]"

# 문장을 토큰으로 변환
tokenized_text = tokenizer.tokenize(text) # text(문장)을 토큰으로 변환해 tokenized_text에 저장합시다!
indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text) # 토큰을 index로 변환하는 코드!

print(tokenized_text) # 토큰이 출력됩니다.
print(indexed_tokens) # 인덱스가 출력됩니다.


['[CLS]', '열', '##여', '##덟', ',', '우리', '##는', '서로', '##의', '이름', '##을', '처음', '불렀', '##다', '.', '그리고', '스물', '하나', ',', '우린', '[MASK]', '을', '했', '##다', '.', '[SEP]']
[2, 1432, 2173, 3542, 16, 3616, 2259, 4084, 2079, 3934, 2069, 3790, 6895, 2062, 18, 3673, 10514, 3657, 16, 8983, 4, 1498, 1902, 2062, 18, 3]


### 1-3. MASK된 위치 확인
- [MASK] 토큰의 위치를 탐지해 해당 위치에서 모델이 단어를 예측하도록 지정합니다.


In [4]:
# 마스킹된 위치 찾기
masked_index = tokenized_text.index("[MASK]") # 토큰화된 결과물에서 index 메서드를 통해 [MASK]의 위치를 확인해봅시다!
print(masked_index)

20


### 1-4. 정수화된 토큰 인덱스를 tensor로 변환 후 모델 예측 실행
- 모델은 **텐서 데이터 구조**를 필요로 하므로, **입력값을 변환**하여 적합한 형식으로 만듦.
   - `torch.tensor(입력값 리스트 또는 입력값 ndarray)`를 통해 tensor 구조로 변환 가능!
- **역전파를 비활성화**하고, 모델이 **각 토큰에 대해 가능한 모든 단어 점수 출력**

In [5]:
# 토큰화된 텍스트 -> 정수인덱스(ID) -> Pytorch 텐서
indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text) # 토큰을 index로 변환하는 코드!
tokens_tensor = torch.tensor([indexed_tokens]) # 입력값(토큰 인덱스 리스트)을 tensor 구조로 변환해봅시다!

# 모델에 토큰 텐서를 전달하고 예측 실행
with torch.no_grad(): # 가중치 업데이트 없으므로 자동미분연산은 끔
    outputs = model(tokens_tensor) # 예측. 모델은 각 토큰위치에 대한 예측 결과를 반환.
    predictions = outputs[0] # logits (각 토큰위치에서 가능한 모든 토큰들의 원시점수)
print(predictions)

tensor([[[ -6.0337,   4.5526,  -5.6312,  ...,  -7.3959,  -7.4220,  -5.8337],
         [ -6.1762,   4.7585,  -7.3973,  ...,  -7.6209, -12.2124,  -6.0998],
         [ -7.4194,   3.9148,  -6.1517,  ...,  -6.8162,  -9.5421,  -3.0473],
         ...,
         [ -7.3761,   8.6972,  -5.4904,  ...,  -8.7202,  -9.1403,  -5.8566],
         [ -5.6347,  10.3884,  -4.3374,  ...,  -8.6900,  -7.4171,  -3.5043],
         [ -5.6652,  10.3407,  -4.4099,  ...,  -8.7521,  -7.4286,  -3.6681]]])


### 1-5. MASK된 토큰 예측
- [MASK] 위치에서 가장 높은 점수를 받은 토큰의 인덱스를 추출하고, 이를 단어로 변환

In [6]:
# 예측된 토큰 확인
predicted_index = torch.argmax(predictions[0, masked_index]).item() # masked_index에 들어갈 것으로 가장 확률이 높은 인덱스
predicted_token = tokenizer.convert_ids_to_tokens([predicted_index])[0]

# 예측된 토큰으로 마스크 채우기
tokenized_text[masked_index] = predicted_token
# 토큰화된 텍스트를 다시 문자열로 변환
filled_text = tokenizer.convert_tokens_to_string(tokenized_text[1:-1])

print("Original:", text)
print("Masked:", tokenized_text)
print("Predicted token:", predicted_token) ; print()

print("Filled sentence:", filled_text)

Original: [CLS] 열여덟, 우리는 서로의 이름을 처음 불렀다. 그리고 스물 하나, 우린 [MASK]을 했다.[SEP]
Masked: ['[CLS]', '열', '##여', '##덟', ',', '우리', '##는', '서로', '##의', '이름', '##을', '처음', '불렀', '##다', '.', '그리고', '스물', '하나', ',', '우린', '사랑', '을', '했', '##다', '.', '[SEP]']
Predicted token: 사랑

Filled sentence: 열여덟, 우리는 서로의 이름을 처음 불렀다. 그리고 스물 하나, 우린 사랑 을 했다.



### 1-6. MASK된 문장 복원
- 예측된 토큰으로 [MASK]를 대체하여 완성된 문장 생성

In [7]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

# BERT 모델과 토크나이저 초기화
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')
model = BertForMaskedLM.from_pretrained('klue/bert-base')

def fill_mask(input_text):
    # 텍스트를 토큰으로 변환
    tokenized_text = tokenizer.tokenize(input_text)
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)

    # 마스킹된 위치 찾기
    masked_index = tokenized_text.index("[MASK]") # 토큰화된 텍스트에서 마스킹된 위치를 찾아봅시다!

    # 토큰화된 텍스트 -> 정수인덱스(ID) -> Pytorch 텐서
    tokens_tensor = torch.tensor([indexed_tokens])

    # 모델에 토큰 텐서를 전달하고 예측 수행
    with torch.no_grad():
        outputs = model(tokens_tensor)
        predictions = outputs[0]

    # 예측된 토큰 확인
    predicted_index = torch.argmax(predictions[0, masked_index]).item() # 위 코드 참고! masked_index에 들어갈 것으로 가장 확률이 높은 인덱스
    predicted_token = tokenizer.convert_ids_to_tokens([predicted_index])[0]

    # 마스크를 채운 문장 반환
    tokenized_text[masked_index] = predicted_token # 토큰화된 텍스트의 마스킹된 위치(인덱스 사용)를 예측된 토큰으로 채워봅시다!
    return tokenizer.convert_tokens_to_string(tokenized_text[1:-1])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


예시 문장 만들어서 MLM을 사용해 보세요! <br>

📌 **아래 예시를 참고해 [CLS](맨 앞), [MASK](채울 부분), [SEP](맨 뒤)를 추가해서 여러분만의 예시 문장을 만들어보세요**

In [8]:
# 예시 1: "[CLS] 열여덟, 우리는 서로의 이름을 처음 불렀다. 그리고 스물 하나, 우린 [MASK]을 했다.[SEP]"
# 예시 2: "[CLS] 권도영은 INSGIHT 학회를 정말 사랑한다. 그는 INSIGHT 학회를 위해 [MASK]를 했다.[SEP]"

input_text = "[CLS] 권도영은 INSGIHT 학회를 정말 사랑한다. 그는 INSIGHT 학회를 위해 [MASK]을 했다.[SEP]"
filled_text = fill_mask(input_text)
print("Filled sentence:", filled_text)

Filled sentence: 권도영은 INSGIHT 학회를 정말 사랑한다. 그는 INSIGHT 학회를 위해 헌신 을 했다.


# 🔍 2. GPT, RAG, Langchain

## 2-1 라이브러리 설치

langchain 관련 라이브러리와, pdf 문서를 읽을수 있는 pypdf 라이브러리를 설치할게요!

In [9]:
%pip install -qU "langchain>=0.3" langchain-community langchain-google-genai langgraph pypdf tiktoken
%pip install -q "chromadb==0.5.23"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.8.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.8.1 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


In [10]:
# 에러 난다면 이 셀 실행해보고 다시 코드 실행

# !pip install langchain-google-genai
# !pip install langchain-google-genai chromadb

<br>
<span style="color:red"> 에러가 난다면 위 코드를 실행해 보신 후, 그 위 코드를 다시 실행해보신 다음에도 아래 코드 진행이 잘 안되면 커널 재시작하고 아래 키값 설정부터 진행해 주세요! </span> <br>   

## 2-2. 키값 설정

# **🚨과제 제출 시 아래 코드 셀은 삭제하고 저장 후, add, commit, push 해주세요🚨**

In [11]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY") # gemini api

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY") # langsmith api
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "practice_langgraph"


## 2-3 pdf 문서 선택 (로드)


여러분들이 활용하고 싶은 아무 pdf 문서를 선택해보세요!

In [12]:
from google.colab import files
import os, re, unicodedata

os.makedirs("/content/pdfs", exist_ok=True)
print("📥 업로드할 PDF 파일을 선택하세요 (여러 개 가능).")
uploaded = files.upload()

name_map = {}
for i, (orig_name, data) in enumerate(uploaded.items(), 1):
    nf = unicodedata.normalize("NFC", orig_name)
    base, ext = os.path.splitext(nf)
    # 한글/영문/숫자/밑줄/대시/점만 남기고 나머지는 _
    safe_base = re.sub(r"[^\w\u3131-\u318E\uAC00-\uD7A3.\-]+", "_", base).strip("._'‘’“”")
    # 파일명 너무 길면 잘라냄 (최대 ~140자 정도로)
    safe_base = safe_base[:120] if len(safe_base) > 120 else safe_base
    new_name = f"doc_{i}_{safe_base}{ext.lower()}"
    with open(os.path.join("/content/pdfs", new_name), "wb") as f:
        f.write(data)
    name_map[new_name] = orig_name

print("✅ 저장 완료 (Colab 파일명 → 원본명 매핑):")
for k, v in name_map.items():
    print(f"  {k}  <=  {v}")


📥 업로드할 PDF 파일을 선택하세요 (여러 개 가능).


Saving ecredible_rating.pdf to ecredible_rating (1).pdf
Saving cmp_grd.pdf to cmp_grd (1).pdf
Saving 2.%20금융안정상황.pdf to 2.%20금융안정상황 (1).pdf
✅ 저장 완료 (Colab 파일명 → 원본명 매핑):
  doc_1_ecredible_rating_1.pdf  <=  ecredible_rating (1).pdf
  doc_2_cmp_grd_1.pdf  <=  cmp_grd (1).pdf
  doc_3_2._20금융안정상황_1.pdf  <=  2.%20금융안정상황 (1).pdf


## 2-4 문서 분할 후 벡터 DB에 저장
로딩한 문서를 분할 할 때,
- chunk_size는 `900`
- chunk_overlap은 `150`
- separators(구분자)는 `["\n\n", "\n", " ", ""]`
으로 설정하여 청킹해주세요!

In [23]:
!pip install -qU transformers sentence-transformers chromadb langchain-huggingface

import time
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings # 최신 버전에 맞게 임포트 경로 변경
from langchain_community.vectorstores import Chroma
import glob

# 1) PDF → Documents
pdf_paths = sorted(glob.glob("/content/pdfs/*.pdf"))
docs = []
for p in pdf_paths:
    try:
        loader = PyPDFLoader(p)
        docs.extend(loader.load())
    except Exception as e:
        print(f"⚠️ 로딩 실패: {p} -> {e}")

print(f"총 문서 조각(페이지 기준): {len(docs)}")

# 2) 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 900, # 청크 사이즈: 문서를 몇 개의 토큰 단위로 나눌 것인지를 정합니다.
    chunk_overlap= 150, # 청크 오버랩: 분할된 끝 부분에서 맥락이 이어질 수 있도록 일부를 겹쳐서 분할합니다.
    separators= ["\n\n", "\n", " ", ""], # 구분자: 엔터, 공백 기준으로 나눔
)
chunks = text_splitter.split_documents(docs)
print(f"총 청크 수: {len(chunks)}")

# 3) 벡터스토어
# 메모리 캐시 충돌 방지를 위해 새로운 경로 사용
persist_dir = "/content/chroma_local_db"

if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)
    print("🗑️ 기존 벡터 DB를 초기화했습니다.")

# 💡 [핵심 변경 사항] Gemini API 대신 무료/빠른 로컬 HuggingFace 임베딩 모델 사용
print("로컬 임베딩 모델을 로드합니다... (API 제한 없음, 매우 빠름!)")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", # 한국어와 영어 모두 성능이 좋은 다국어 모델
    model_kwargs={'device': 'cuda'} # 할당받은 Colab GPU 사용
)

# 로컬 모델이므로 API 제한이 없어 한 번에 처리 가능합니다. time.sleep이 전혀 필요 없습니다.
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_dir
)

retriever = vectordb.as_retriever(search_kwargs={"k": 5})

print("✅ 인덱싱 완료")


총 문서 조각(페이지 기준): 210
총 청크 수: 524
로컬 임베딩 모델을 로드합니다... (API 제한 없음, 매우 빠름!)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ 인덱싱 완료


## 2-5 RA<U>Generate</U> : 생성 부분 선언

PROMPT 부분은 여러분들이 원하시는대로 입력해도 좋습니다!

🤥 영어로 입력하면 모델이 더 잘 이해한대요

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

PROMPT = ChatPromptTemplate.from_template(
"""Answer the following question in Korean based ONLY on the provided context (extracted from the uploaded PDF).
If the answer is not contained within the context, state that you do not know.
Summarize the key evidence and provide the specific source (file name/page number) next to each piece of evidence.

[질문]
{question}

[컨텍스트]
{context}
"""
)

rag_chain = (PROMPT | llm | StrOutputParser())


## 2-6 RAG 파이프라인 langgraph로 구성

벡터db에서 검색하는 것이 <U>Retrieve</U>AG이고,

검색 결과를 생성 단계에 던져주는 것이 R<U>Augment</U>G입니다!

In [25]:
from typing import TypedDict, List
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

# format_docs의 역할은 문서를 포맷팅해서 모델에게 제공할 'context'를 정리하는 역할입니다!
def format_docs(docs):
    # 파일명/페이지를 메타데이터에 담아두는 PyPDFLoader 기본값 사용
    lines = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "")
        page = d.metadata.get("page", None)
        tag = f"{os.path.basename(src)}"
        if page is not None:
            tag += f" p.{page+1}"
        lines.append(f"[{i}] {tag}\n{d.page_content}")
    return "\n\n".join(lines)

# 그래프 선언
class QAState(TypedDict):
    question: str
    retrieved: List[Document]
    answer: str

def node_retrieve(state: QAState):
    q = state["question"]
    topk = retriever.invoke(q)
    return {"retrieved": topk}

def node_generate(state: QAState):
    q = state["question"]
    ctx = format_docs(state["retrieved"])
    ans = rag_chain.invoke({"question": q, "context": ctx})
    return {"answer": ans}

# --------- 그래프 구성 ---------
# 노드끼리 연결하기
workflow = StateGraph(QAState)
workflow.add_node("retrieve", node_retrieve) # 어떤 노드가 추가되어야 할까요?
workflow.add_node("generate", node_generate) # 어떤 노드가 추가되어야 할까요?

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate") # 어떤 노드와 어떤 노드가 연결되어야 할까요?
workflow.add_edge("generate", END)

app = workflow.compile()
print("✅ LangGraph 컴파일 완료")


✅ LangGraph 컴파일 완료


## 2-7 실제 구동해보기

question에 여러분들이 입력하고 싶은 질문을 넣어보세요!

In [26]:
question = "기업신용평가에서 중요한건 무엇인가?."

final = app.invoke({"question": question})
print("▼ 답변")
print(final["answer"])


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


▼ 답변
기업신용평가에서 중요한 것은 기업의 신용도에 영향을 미치는 재무 및 비재무 요소를 분석하고 평가하는 것입니다.

**핵심 증거:**

*   기업신용평가는 시장환경변동상황, 산업의 특성 및 동향분석, 그리고 기업의 재무상태, 경영성과, 경영 관리상태 등 그 기업의 특수한 자료를 수집 및 분석하는 과정이 선행되어야 합니다. (doc_1_ecredible_rating_1.pdf p.1, doc_1_ecredible_rating.pdf p.1)
*   기업의 신용능력을 평가하기 위해 기업체의 환경위험, 경영위험, 영업위험, 재무위험 등 기업의 신용도에 영향을 미치는 재무 및 비재무 요소를 분석, 평가합니다. (doc_1_ecredible_rating_1.pdf p.1, doc_1_ecredible_rating.pdf p.1)
*   **재무평가 요소**로는 안정성(유동비율, 자기자본비율, 부채비율, 차입금의존도 등), 수익성(매출액영업이익률, 매출액순이익률 등), 활동성(총자산회전율, 재고자산회전기간, 매출채권회전율 등), 성장성(총자산증가율, 매출액증가율, 순이익증가율 등)이 있습니다. (doc_1_ecredible_rating_1.pdf p.12)
*   **비재무평가 요소**로는 경영위험(경영진 대표자 신용도, 경영역량, 설립업력), 영업위험(판매위험, 거래처 신용도), 재무위험(자금능력, 여신만기구조, 자금조달여력, 재무전망, 수익전망, 현금흐름전망, 재무건전성, 자본구조, 질적재무위험) 등이 있습니다. (doc_1_ecredible_rating_1.pdf p.12)


In [35]:
question = "문서 별로 중요 내용을 설명해줘"

final = app.invoke({"question": question})
print("▼ 답변")
print(final["answer"])


▼ 답변
제공된 컨텍스트에 따라 문서별 중요 내용은 다음과 같습니다.

**doc_1_ecredible_rating.pdf**
*   **p.5:** 평가자료 중 "기초자료"는 반드시 요청해야 하며, 재무자료는 국세청에 결산신고가 확정된 재무제표만을 인정하여 반영합니다. 외감기업(상장포함)의 경우 외부감사인의 감사보고서가 필수자료입니다.
*   **p.6:**
    *   외감법인의 경우 감사보고서로 반영합니다. (doc_1_ecredible_rating.pdf p.6)
    *   비외감 및 소기업은 결산 신고가 완료된 확정 재무제표가 필수자료이며, 원가명세서 자료제시 거부 시 원가명세서 없이 진행합니다. (doc_1_ecredible_rating.pdf p.6)
    *   조기결산 및 가결산 평가 시 외감기업은 제외되며, 필수자료(재무제표, 법인세납부영수증, 법인세 신고서 접수증, 법인세액조정계산서 등)가 요구됩니다. (doc_1_ecredible_rating.pdf p.6)
    *   신설기업, 간편장부작성기업 등 재무제표 제출이 불가능한 기업은 기업소개서, 사업계획서 등 비재무 평가항목으로만 평가를 진행합니다. (doc_1_ecredible_rating.pdf p.6)
    *   합병기업은 직전 사업연도의 합병기업 재무제표로 평가하며, 법인전환기업은 개인사업자 재무제표로 평가하고, 법인전환기업의 요건(개인사업자 폐업, 대표자 동일, 포괄양수도 처리)이 명시되어 있습니다. (doc_1_ecredible_rating.pdf p.6)
    *   본/지점기업은 합산 재무제표로 평가를 진행합니다. (doc_1_ecredible_rating.pdf p.6)
    *   평가자료(재무자료 및 필수제출자료) 제시를 거부할 경우 평가업무를 중단 또는 거부할 수 있으나, 평가기업이 소명자료 제출을 통해 평가진행을 요청하는 경우 타당성 검토 후 평가를 진행할 수 있습니다. (doc_1_ecredible_rating.pdf p.6)

**doc_1_ecr

In [36]:
question = "재무제표가 중요한 이유가 뭐야?"

final = app.invoke({"question": question})
print("▼ 답변")
print(final["answer"])


▼ 답변
제공된 컨텍스트에 따르면 재무제표가 중요한 이유는 다음과 같습니다.

재무제표는 기업신용평가 시 기업의 **재무상태, 경영성과, 재무위험 등을 분석하고 평가하여 신용도 및 신용거래능력을 판단하는 데 필요한 '기초자료'이자 '필수자료'**이기 때문에 중요합니다.

*   기업신용평가는 기업의 재무상태, 경영성과 등을 분석하는 과정이 선행되어야 하며, 이를 **기초자료**로 대상기업의 신용도 및 신용거래능력을 판단합니다. (doc_1_ecredible_rating_1.pdf p.1)
*   기업의 신용능력을 평가하기 위해 기업의 신용도에 영향을 미치는 **재무위험 등 재무 요소를 분석, 평가**하는 데 사용됩니다. (doc_1_ecredible_rating_1.pdf p.1)
*   비외감 및 소기업의 경우 결산 신고가 완료된 확정 재무제표가 **필수자료**이며, 조기결산 및 가결산 평가기업의 경우에도 재무제표가 **필수자료**로 명시되어 있습니다. (doc_1_ecredible_rating.pdf p.6, doc_1_ecredible_rating_1.pdf p.6)
*   재무제표 제출이 불가능한 기업의 경우, **비재무 평가항목으로만 평가가 진행**된다고 명시되어 있어, 재무제표가 재무적 평가에 필수적임을 시사합니다. (doc_1_ecredible_rating.pdf p.6, doc_1_ecredible_rating_1.pdf p.6)
*   당사는 국세청에 결산신고가 확정된 재무제표만을 인정하여 반영하는 **재무자료**의 핵심입니다. (doc_1_ecredible_rating.pdf p.5, doc_1_ecredible_rating_1.pdf p.5)


# 🔍 3. sLM

Copyright 2024 Google LLC.

## 3-1. sLM(Gemma) 로드
### sLM(small Language Model)의 대표 모델인 Gemma를 로드 해봅시다!

### (1) 사전 준비 사항
### 1. HuggingFace 로그인

https://huggingface.co/

이 링크로 들어가서 HuggingFace에 로그인합니다.(계정이 없다면 회원가입 해주세요!)

### 2. 토큰 발급받기

https://huggingface.co/settings/tokens

- 이 링크로 들어가서 'Create new token'을 클릭
- Token type: 'Read'로 해주세요.
- Token name: 아무거나 상관 없습니다.
- (⭐ 매우 중요!) 토큰 만드시고 **반드시 토큰 문자열 (hf_... 로 시작함)을 복사해주세요!**  
한 번만 볼 수 있습니다!

### 3. google colab에서 토큰 등록하기
- 왼쪽에 열쇠 모양 버튼(보안 비밀)을 눌러주세요.
- 이름: colab-gemma
- 값: 복사해둔 토큰 문자열(hf_...로 시작함)
- 노트북 액세스 : 스위치 키기(파란색)

### 4. gemma-2b-it 모델 사용 약관 동의

https://huggingface.co/google/gemma-2b-it

- (⭐ 매우 중요!) 반드시 2번 단계에서 토큰을 발급받았던 바로 그 Hugging Face 계정으로 로그인된 상태에서 위 링크로 들어가야 합니다.
- 링크로 들어가서 'Access Gemma on Hugging Face' 부분의 약관을 읽고 동의 버튼을 클릭해 주세요.
- "You have been granted access to this model"이라는 메시지가 뜨면 성공입니다.

### (2) 필요한 툴 설치

In [1]:
!pip install -U bitsandbytes
!pip install --upgrade -q transformers huggingface_hub peft \
  accelerate bitsandbytes datasets trl

<br>
<span style="color:red"> 만약 아래부터 잘 코드가 돌아가지 않으면, 커널 재시작하고 아래 코드부터 진행해 주세요! </span> <br>  

In [2]:
# HuggingFace에 로그인합니다.

from huggingface_hub import login
from google.colab import userdata

# 따음표 안에 본인의 보안 비밀 키의 이름을 적습니다.(여기선 HF_TOKEN)
try:
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print(f"Error logging in to Hugging Face: {e}")
    print("Please make sure you have added your Hugging Face token to Colab secrets with the name 'HF_TOKEN'")

In [3]:
!pip install -U transformers
# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("google/gemma-4-E2B-it")
model = AutoModelForImageTextToText.from_pretrained("google/gemma-4-E2B-it")

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

In [5]:
# 모델을 로드합니다.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 계산 부하를 줄이고 추론 속도를 높이기 위해 '양자화'합니다.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

# 권한 승인이 필요 없는 완전 오픈 모델로 변경합니다 (Gemma 우회)
# 성능이 좋고 가벼운 Qwen 모델을 사용합니다.
model_id = "Qwen/Qwen1.5-1.8B-Chat"

# 토크나이저를 로드합니다.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 모델을 로드합니다.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # 양자화(quantization)
    device_map={"":"cuda:0"}
)

# GPU를 사용하도록 설정합니다.
device = "cuda"

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

## 3-2. sLM 활용
- sLM은 LLM을 경량화한 모델이기 때문에 언어 모델의 기능인 질의응답(추론), 요약, 분류(감정분석) 등이 가능합니다.
- **속도는 비교적 빠르지만, 성능이 그닥 좋지 않다는 것을 확인하실 수 있습니다.**

### Gemma 활용 예시 (Common Use Cases)


### (1) 질의응답(Reasoning)

#### 원하는 질문을 한 번 넣어보세요!

In [6]:
print("--- 예시 1: 질의응답 (Reasoning) ---")

# content : '~~' 따음표 안에 질문을 써주시면 됩니다.
# 하고 싶은 질문 아무거나 넣어보세요!

# "role": "user" 입니다.

chat = [
    # Gemma 모델의 채팅 형식에 맞춰 'role'과 'content'를 지정합니다.
    { "role": "user", "content": "AI가 미래 금융에서 어떻게 활용될 수 있는지 알려줘" } # **** user
]

# 1. 딕셔너리 리스트('chat')를 Gemma 모델이 이해할 수 있는 공식 프롬프트 문자열로 변환합니다.
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# 2. 변환된 프롬프트 문자열을 모델이 입력받을 수 있는 토큰 ID 텐서로 변환하고, GPU('device')로 보냅니다.
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# 3. 토큰화된 입력을 모델에 전달하여 텍스트 ""생성""을 시작합니다 (최대 256개의 새 토큰 생성)
# 생성을 위해 'generate' 함수를 사용합니다.
outputs = model.generate(**inputs, max_new_tokens=256) # **inputs에서 **는 빈칸이 아닙니다.

# 4. 'outputs'에는 '입력'과 '답변'이 모두 포함되어 있습니다.
# 따라서 원본 입력 토큰의 길이(입력, 'inputs.input_ids.shape[1]')를 기준으로, 그 이후에 '새로 생성된' 토큰(답변)만 추출합니다.
new_tokens = outputs[0][inputs.input_ids.shape[1]:]

# 5. 추출된 토큰 ID 리스트를 다시 사람이 읽을 수 있는 문자열로 ""디코딩""합니다.
# 디코딩을 위해 'decode' 함수를 사용합니다.
text = tokenizer.decode(new_tokens, skip_special_tokens=True)

# 6. 최종 생성된 답변 텍스트를 화면에 출력합니다.
print(text)

--- 예시 1: 질의응답 (Reasoning) ---
AI는 금융 분야의 전략적 역할을 합니다. 다음은 AI가 금융에서 어떻게 활용될 수 있는 몇 가지 예시입니다:

1. 자동화 업무: AI는 금융 거래, 계약 관리, 고객 서비스, 데이터 저장 및 분석 등의 작업을 자동화합니다. 예를 들어, AI는 대출 카드 발행을 수행하는 프로세스를 자동화하고, 자동화한 보조금 부담 감소와 고객 불만 해결을 수행하는 데 사용됩니다. 이러한 기술은 인력과 시간 절약을 위해 필요한 효율적인 작업을 수행하며, 비용 절감도 있습니다.

2. 자동화 대출 및 입자: AI는 개인별한 금융 상황을 모델링하여 대출 및 입자 처리에 사용할 수 있습니다. 예를 들어, AI는 개인 간의 금융 정보 소비 및 은유를 분석하여 개인간 거래 및 가격 측면을 최대化시키며, 정확한 거래 내역 추적 및 탐색이 가능해집니다. 이는 개인


### (2) 요약 (Summarization)

#### 원하는 글을 넣어서 요약해 보세요!

In [7]:
print("\n--- 예시 2: 요약 (Summarization) ---")

# text_to_summarize = """~~~""" 따음표 안에 요약하고 싶은 글을 넣어주시면 됩니다.""
# 요약하고 싶은 글 아무거나 넣어보세요!

text_to_summarize = """

송하린 기자 = 코스피가 사상 처음 8,000을 돌파한 당일 6% 급락하며 고점 우려를 키웠지만, 증권가에서는 실적 모멘텀에 근거해 자산배분상 주식 선호 의견을 유지한다는 의견이 나왔다.

17일 최재원 키움증권 연구원은 "국내 증시의 폭발적인 상승 흐름은 수익률 측면에서는 긍정적이나 자산배분 전략 측면에서는 고민할 지점을 만들고 있다"며 "지난해 글로벌 증시 내 비중이 1.8%였던 한국 증시는 현재 2.86%까지 상승했다"고 말했다.

최 연구원은 "자국 편향이 있는 국내 자산배분 포트폴리오의 경우 국내주식이 목표 비중 범위를 큰 폭으로 상회했을 가능성이 높다"며 "다만 산업 구조 재편 및 국내 증시 리레이팅 흐름이 여전히 진행 중인 점을 감안해 단기적인 레이지 이탈을 유보하며 리밸런싱 시점을 유동적으로 관리할 시점"이라고 판단했다.

그 첫 번째 근거로는 하이퍼 스케일러들이 설비투자(CAPEX) 투자를 확대하고 있는 점을 들었다.

최 연구원은 "반도체뿐 아니라 전력 설비, 냉각 시스템 등 인공지능(AI) 인프라 전반에 걸쳐 장기적인 수요 가시성이 확보됐다"며 "그 결과 AI 관련 산업을 주도하는 미국을 비롯해 신흥국 내 한국, 대만, 중국 등 아시아 국가들의 올해 실적 성장률 전망은 상향되며 실적 모멘텀을 강화하는 배경"이라고 설명했다.

국내증시 자기자본이익률(ROE)이 주요국을 크게 상회하는 부분도 추가적인 주가 상승 기대감을 지속하는 이유다.

최 연구원은 "한국 증시의 수익성은 주요국 중 가장 높은 수준"이라며 "최근 상승으로 주가순자산비율(PBR)은 신흥국을 소폭 상회한 2.23배까지 높아졌다"고 분석했다.

그는 "단기간 코리아 디스카운트에 대한 부분이 전적으로 해소될 것으로 기대하기는 어렵지만, AI 밸류체인을 주도하고 있는 미국, 대만 등의 멀티플을 고려하면 추가적인 주가 상승 개선 기대가 유효하다"고 말했다.



"""

# content : '~~~'에 요청사항을 적으시면 됩니다. 여기서는 위 글을 세 문장으로 요약해 달라고 요청했습니다.
# "role": "user" 입니다.

chat = [
    { "role": "user", "content": f"Could you summarize the following text in three sentence?\n\nText:\n{text_to_summarize}" }
]

# 여기부터는 각 사례 모두 코드가 동일합니다.

# 1. 딕셔너리 리스트('chat')를 Gemma 모델이 이해할 수 있는 공식 프롬프트 문자열로 변환합니다.
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# 2. 변환된 프롬프트 문자열을 모델이 입력받을 수 있는 토큰 ID 텐서로 변환하고, GPU('device')로 보냅니다.
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# 3. 토큰화된 입력을 모델에 전달하여 텍스트 ""생성""을 시작합니다 (최대 256개의 새 토큰 생성).
# 생성을 위해 'generate' 함수를 사용합니다.
outputs = model.generate(**inputs, max_new_tokens=256) # **inputs에서 **는 빈칸이 아닙니다.

# 4. 'outputs'에는 '입력'과 '답변'이 모두 포함되어 있습니다.
# 따라서 원본 입력 토큰의 길이(입력, 'inputs.input_ids.shape[1]')를 기준으로, 그 이후에 '새로 생성된' 토큰(답변)만 추출합니다.
new_tokens = outputs[0][inputs.input_ids.shape[1]:]

# 5. 추출된 토큰 ID 리스트를 다시 사람이 읽을 수 있는 문자열로 ""디코딩""합니다.
# 디코딩을 위해 'decode' 함수를 사용합니다.
text = tokenizer.decode(new_tokens, skip_special_tokens=True)

# 6. 최종 생성된 답변 텍스트를 화면에 출력합니다.
print(text)


--- 예시 2: 요약 (Summarization) ---
South Korean stock market increased by 6% on Sunday, driven by a surge in global exchange rates, with investors expressing concern about short-term volatility but maintaining their support for diversified investment portfolios and corporate earnings distribution strategies. The research showed that domestic index funds' exposure to the capital market was significantly higher than before, leading to temporary losses. The study also pointed out that long-term growth prospects of South Korea and other Asian countries were promising, as they saw an increase in domestic equity returns. Investors are optimistic about future gains due to increasing interest in artificial intelligence (AI) sectors led by the United States, among others, and considering multiple financial instruments such as multi-family assets. The recent appreciation of PBR, which reflects market dominance of Korea, is expected to stabilize over a short period but has potential improvement, a

### (3) 분류 (Classification)

원하는 문장을 넣고 감정 분석을 해보세요!

In [8]:
print("\n--- 예시 3: 분류 (Classification) ---")

# content: '~~~' 따음표 안에 요청사항을 적으시면 됩니다.
# 여기서는 긍정, 부정, 중립으로 감정분석을 요청했습니다.

# \n\nText: 여기에 감정분석할 문장을 적어주시면 됩니다.
# 원하는 문장을 넣고 감정 분석을 해보세요!

# "role": "user" 입니다.

chat = [
    { "role": "user", "content": "Classify the text into neutral, negative, or positive. Generate only the class, nothing else.\n\nText: 이영화 미쳤다 (영화 리뷰)" }
]

# 1. 딕셔너리 리스트('chat')를 Gemma 모델이 이해할 수 있는 공식 프롬프트 문자열로 변환합니다.
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# 2. 변환된 프롬프트 문자열을 모델이 입력받을 수 있는 토큰 ID 텐서로 변환하고, GPU('device')로 보냅니다.
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# 3. 토큰화된 입력을 모델에 전달하여 텍스트 ""생성""을 시작합니다 (최대 256개의 새 토큰 생성).
# 생성을 위해 'generate' 함수를 사용합니다.
outputs = model.generate(**inputs, max_new_tokens=256) # **inputs에서 **는 빈칸이 아닙니다.

# 4. 'outputs'에는 '입력'과 '답변'이 모두 포함되어 있습니다.
# 따라서 원본 입력 토큰의 길이(입력, 'inputs.input_ids.shape[1]')를 기준으로, 그 이후에 '새로 생성된' 토큰(답변)만 추출합니다.
new_tokens = outputs[0][inputs.input_ids.shape[1]:]

# 5. 추출된 토큰 ID 리스트를 다시 사람이 읽을 수 있는 문자열로 ""디코딩""합니다.
# 디코딩을 위해 'decode' 함수를 사용합니다.
text = tokenizer.decode(new_tokens, skip_special_tokens=True)

# 6. 최종 생성된 답변 텍스트를 화면에 출력합니다.
print(text)


--- 예시 3: 분류 (Classification) ---
Positive.



#### sLM 활용사례 결과 요약

- sLM은 질의응답이나 요약은 어느 정도 수행하지만, 퀄리티가 아쉽습니다.
- 감정 분석에서 '너무 재미가 없다'를 중립으로 판단하는 것으로 보아 분류 작업에서도 한계가 나타납니다.


- 이렇게 범용적인 활용에서는 한계가 나타나지만, 미세조정(Fine-Tuning)을 통해 특정 작업에 특화하면 해당 작업에는 좋은 성능을 내면서도 여전히 경량화된 모델의 장점을 누릴 수 있는 것이 sLM의 특징입니다!

### 🤩 수고하셨습니다 🤩
# 🚨 **2-2의 API KEY 셀을 지우고 과제 제출해주세요!!** 🚨